In [7]:
import travel_agent.config  # loads .env and sets API key
from agents import Runner, trace
from travel_agent.agents.intake import intake_agent
from travel_agent.agents.visa import visa_agent
from travel_agent.agents.activities import activities_agent

## 1. Intake: multi-turn chat


In [8]:
history = []

async def say(msg):
    global history
    result = await Runner.run(intake_agent, history + [{'role': 'user', 'content': msg}])
    history = result.to_input_list()
    out = result.final_output
    print('missing:', out.missing_fields)
    print('assistant:', out.next_question)
    return out

out = await say('I want to visit Japan with 2 friends next month for 7 days, budget 6000 USD')

missing: ['trip_type', 'passport_country', 'origin', 'start_date']
assistant: What is the purpose of your trip? Leisure or business?


In [9]:
out = await say('I hold a French passport, flying from Miami. Food and temples.')

missing: ['start_date']
assistant: What is your planned start date?


In [10]:
trip = out.trip
print(trip.model_dump_json(indent=2))
print('complete:', out.is_complete)

{
  "trip_type": "leisure",
  "passport_country": "France",
  "origin": "Miami",
  "destination": "Japan",
  "start_date": null,
  "end_date": null,
  "num_days": 7,
  "travelers": 3,
  "budget_usd": 6000.0,
  "interests": [
    "Food",
    "temples"
  ],
  "meeting_address": null,
  "needs_car_rental": null
}
complete: False


## 2. Visa agent


In [11]:
with trace('Day4 visa'):
    r = await Runner.run(visa_agent, f'Passport: {trip.passport_country}. Destination: {trip.destination}.')
print(r.final_output.model_dump_json(indent=2))

{
  "passport": "FRA",
  "destination": "JPN",
  "requirement": "visa_free",
  "allowed_days": 90,
  "summary": "French passport holders can enter Japan without a visa and stay for up to 90 days visa-free. This information is sample data and should be verified with official sources before travel.",
  "source": "mock"
}


## 3. Activities agent


In [12]:
prompt = (
    f'Destination: {trip.destination}. Dates: {trip.start_date} to {trip.end_date}. '
    f'Travelers: {trip.travelers}. Budget USD: {trip.budget_usd}. '
    f'Interests: {trip.interests}. Trip type: {trip.trip_type}.'
)
with trace('Day4 activities'):
    r = await Runner.run(activities_agent, prompt)
plan = r.final_output
print(plan.weather_summary)
for a in plan.activities:
    print('-', a.name, '|', a.category, '|', a.estimated_cost_usd)

Weather was not checked as dates were not provided.
- Sample Temple | tourism.sights | None
- Sample Museum | entertainment.museum | None
- Sample Temple | tourism.sights | None
- Sample Museum | entertainment.museum | None
- Sample Temple | tourism.sights | None
- Sample Museum | entertainment.museum | None
